# Classification and representative evaluation

This notebook reads the archived grouped evaluation and operational model settings. It constructs the 17 predictors from the development spectra and checks the human reference totals. Model fitting itself is performed by the terminal model or complete route.

In [1]:
from pathlib import Path
import os,sys,json
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'config.json').exists())
sys.path.insert(0,str(ROOT))
WORK = Path(os.environ.get('RBC_OUTPUT',ROOT/'work')).resolve()
os.environ['RBC_OUTPUT']=str(WORK)
import pandas as pd
from IPython.display import display,SVG
from dados import CFG


## Evaluate candidates and the selected representatives

The predictors are ten reflectances divided by the spectrum’s Euclidean norm, six spectral indices and log(1+norm). Five outer partitions group candidates by class and date to evaluate the procedure. Four inner partitions choose among four tree configurations and select the threshold with the highest sensitivity subject to positive precision ≥0.95 and at least 30 accepted examples.

Each outer model classifies every candidate in its reserved groups. The accepted candidate with the highest PPI is then selected; a tie is resolved by the lowest canonical pixel index. Human judgments assess those selected representatives. The final operational model is fitted on 2,422 development examples. The 58 evaluation-only judgments remain outside that fit. The 95% compatibility target applies to the overall representative evaluation.

In [2]:
from nucleo import atributos
training=pd.read_csv(WORK/'model/training.csv')
X,names=atributos(training);assert X.shape==(2422,17)
display(pd.Series(names,name='Spectral predictors'))
display(pd.read_csv(WORK/'evaluation/candidate_external_metrics.csv'))
reference=pd.read_csv(WORK/'evaluation/reference.csv')
display(reference.groupby('classe').evaluation_label.value_counts())
assert reference.evaluation_label.eq('compatible').sum()==502
print('External representative compatibility: 502/519 = 96.72%')
display(json.loads((WORK/'model/model.json').read_text())['choice'])


0      forma_B02
1      forma_B03
2      forma_B04
3      forma_B05
4      forma_B06
5      forma_B07
6      forma_B08
7      forma_B8A
8      forma_B11
9      forma_B12
10          ndvi
11           evi
12          ndwi
13          ndre
14          ndmi
15           bsi
16    log_brilho
Name: Spectral predictors, dtype: object

,fold,threshold,chosen,n_training,n_test,n_test_groups,selected,ap,roc_auc,tn,fp,fn,tp,precision,recall,specificity
0,1,0.743876,compact,1938,484,104,104,0.990220,0.997426,378,3,11,92,0.968421,0.893204,0.992126
1,2,0.839150,moderate,1938,484,104,104,0.984262,0.994484,378,2,10,94,0.979167,0.903846,0.994737
2,3,0.610412,compact,1942,480,104,104,0.958916,0.990516,368,6,5,101,0.943925,0.952830,0.983957
3,4,0.739747,regularized,1933,489,104,104,0.978425,0.995820,384,3,8,94,0.969072,0.921569,0.992248
4,5,0.623638,moderate,1937,485,103,103,0.982469,0.995610,377,8,5,95,0.922330,0.950000,0.979221


classe  evaluation_label
1       compatible          173
2       compatible          164
        incompatible          9
9       compatible          165
        incompatible          8
Name: count, dtype: int64

External representative compatibility: 502/519 = 96.72%


{'threshold': 0.7542582655199784,
 'recall': 0.9300970873786408,
 'precision': 0.952286282306163,
 'predicted_positive': 503,
 'ap': 0.9785349648543274,
 'name': 'regularized',
 'order': 2}

## Read the two evaluations

Candidate metrics describe predictions on labeled examples reserved from each outer fit. The representative reference describes the output of classification followed by highest-PPI selection: 502 of 519 representatives were judged compatible (96.72%). The printed operational choice comes from internal selection on the full development base; it is used to build the time series.

## References

Friedman (2001), gradient boosting; Pedregosa et al. (2011), scikit-learn (full citations in `REFERENCES.md`). Cawley and Talbot (2010): https://www.jmlr.org/papers/v11/cawley10a.html . Threshold selection: https://scikit-learn.org/1.6/modules/classification_threshold.html